<a href="https://colab.research.google.com/github/Jmian1520/Data-Sceince-Project/blob/main/MLPC_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files

In [ ]:
#Read CSV file
df = pd.read_csv("healthcare-dataset-stroke-data.csv")

# Display the dataset first 5 rows
print("First 5 rows of the dataset:")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'healthcare-dataset-stroke-data.csv'

In [ ]:
#Display the data basic information & statistic
print("Dataset Information:")
df.info()

print("\nBasic Statistics (Before Cleaning):")
df.describe(include='all')

In [ ]:
# Checking Missing values
print("Missing values:")
df.isnull().sum()

In [ ]:
# Replace missing value
df['bmi'] = df['bmi'].fillna(df['bmi'].median())

#check missing value again
print("Missing values(after cleaning):")
df.isnull().sum()

In [ ]:
# Check for duplicate row
print("Duplicate Row:")
print(df.duplicated().sum())

# Check for duplicates based on unique identifier
print("\nDuplicate id:")
print(df['id'].duplicated().sum())

In [ ]:
#Handle Outlier
num_cols = ['age', 'avg_glucose_level', 'bmi']

# Apply box plot
plt.figure(figsize=(15, 10))
sns.boxplot(data=df[num_cols])
plt.title('Box Plot for Numerical Features')
plt.ylabel('Value')
plt.show()

**EDA**

In [ ]:
#Distribution of target variable
#Countplot
sns.countplot(x='stroke', data=df)
plt.title('Distribution of Stroke vs Non-Stroke Cases')

#Apply pie to see precentage
plt.figure(figsize=(10, 6))
df['stroke'].value_counts().plot(kind='pie', autopct='%1.1f%%', colors=['lightblue','lightpink'])
plt.title('\nStroke Distribution')
plt.ylabel('')
plt.show()

In [ ]:
# Demographic Analysis
# Demographic categorical variables
demographic_cols = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']

# Set up countplot
plt.figure(figsize=(12, 10))

for i, col in enumerate(demographic_cols, 1):
    plt.subplot(2, 3, i)
    sns.countplot(x=col, hue='stroke', data=df, palette='pastel', edgecolor='black')
    plt.title(f'{col} Distribution')
    plt.xlabel(col.capitalize())
    plt.ylabel('Count')
    plt.xticks(rotation=25)

plt.tight_layout(pad=2.0)
plt.show()

# Demographic numeric variable
print("Age distribution:\n")
sns.histplot(data=df, x='age', hue='stroke', bins=30, kde=True)

In [ ]:
# Symptoms Indicator Analysis
# Categorical variables
symp_cols = ['hypertension', 'heart_disease']

# Set up countplot
plt.figure(figsize=(10, 6))
for i, col in enumerate(symp_cols, 1):
    plt.subplot(1 ,2 , i)
    sns.countplot(x=col, hue='stroke', data=df, palette='cool', edgecolor='black')
    plt.title(f'{col} Distribution')
    plt.xticks(rotation=25)
plt.tight_layout(pad=2.0)
plt.show()

# Numeric features
num_symp_cols = ['bmi','avg_glucose_level']

# Apply histograms
df[num_symp_cols].hist(bins = 15,figsize = (15,6),layout=(3,3))
plt.tight_layout(pad=1.0)
plt.show()

Correlation analysis

In [ ]:
from scipy.stats import chi2_contingency
import pandas as pd

# Define categorical features
cat_features = ['gender', 'ever_married', 'work_type',
                'Residence_type', 'smoking_status',
                'hypertension', 'heart_disease']

chi_square_results = []

for col in cat_features:
    contingency = pd.crosstab(df[col], df['stroke'])
    chi2, p, dof, expected = chi2_contingency(contingency)
    chi_square_results.append({
        'Feature': col,
        'Chi2': round(chi2, 2),
        'p-value': round(p, 4)
    })

chi_square_df = pd.DataFrame(chi_square_results)
print(chi_square_df)


In [ ]:
# Point-biserial correlation
from scipy.stats import pointbiserialr

continuous_vars = ['age', 'avg_glucose_level', 'bmi']
for var in continuous_vars:
    corr, p_value = pointbiserialr(df[var], df['stroke'])
    print(f"{var}: Point-biserial correlation = {corr:.3f}, p-value = {p_value:.4f}")

**Encoding**

In [ ]:
# Check gender distribution
print(df['gender'].value_counts())

# Drop the single 'Other' gender record
df = df[df['gender'] != 'Other']
print("\n", df['gender'].value_counts())

In [ ]:
# Binary encoding for Gender, Residence type, even married
# manually set "Male" > 1 and "Female" > 0
gender_map = {'Male': 1, 'Female': 0}
#Apply binary encoding to gender column and drop the original data
df['Gender_Encoded'] = df['gender'].map(gender_map)
df.drop("gender", axis=1, inplace=True)

# manually set "Urban" > 1 and "Rural" > 0
residence_map = {'Urban': 1, 'Rural': 0}
#Apply binary encoding to Residence type column and drop the original data
df['Residence_type_Encoded'] = df['Residence_type'].map(residence_map)
df.drop("Residence_type", axis=1, inplace=True)

# manually set "Yes" > 1 and "No" > 0
ever_married_map = {'Yes': 1, 'No': 0}
#Apply binary encoding to even married column and drop the original data
df['ever_married_encoded'] = df['ever_married'].map(ever_married_map)
df.drop('ever_married', axis=1, inplace=True)

# Display the updated dataframe
print('After Encoded')
print(df.shape)
print(df.head())

In [ ]:
# One-hot encoding for work_type
df = pd.get_dummies(df, columns=['work_type'], drop_first=True)

# Display the updated dataframe
print('After Encoded')
print(df.shape)
print(df.head())

In [ ]:
# Ordinal Encoding for smoking_status
from sklearn.preprocessing import OrdinalEncoder

#set the order status
smoking_status_order = [['Unknown','never smoked', 'formerly smoked', 'smokes']]
ordinal_encoder = OrdinalEncoder(categories=smoking_status_order)

#Apply ordinal encoder
df['Smoking_status_Encoded'] = ordinal_encoder.fit_transform(df[['smoking_status']])
df = df.drop('smoking_status', axis=1)

# Check the result
print('After Encoding')
print(df.shape)
print(df.head())


In [ ]:
from sklearn.preprocessing import StandardScaler

#Initialize Standard Scaler
scaler = StandardScaler()
# Select all numerical feautures and apply standard scaler
df[['age','avg_glucose_level','bmi']] = scaler.fit_transform(df[['age','avg_glucose_level','bmi']])

# Display after standardisation
print('After Standardisation')
print(df.head())

In [ ]:
from sklearn.model_selection import train_test_split

# Define features and target
X = df.drop(['stroke','id'], axis=1)
y = df['stroke']

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,stratify=y,random_state=42)

print("Training set size:", X_train.shape)
print("Testing set size:", X_test.shape)

In [ ]:
from imblearn.over_sampling import SMOTE
from collections import Counter

# Apply SMOTE to train test
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

print("After SMOTE:", Counter(y_train_res))
print("Original shape:", X_train.shape)
print("New shape after SMOTE:", X_train_res.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
log_reg.fit(X_train_res, y_train_res)
y_pred_lr = log_reg.predict(X_test)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
rf.fit(X_train_res, y_train_res)
y_pred_rf = rf.predict(X_test)


In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier(random_state=42,
                    scale_pos_weight=10,
                    learning_rate=0.05,
                    max_depth=5,
                    n_estimators=200)
xgb.fit(X_train_res, y_train_res)
y_pred_xgb = xgb.predict(X_test)

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

# apply gridsearchcv to logistic regression
lr_param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'class_weight': ['balanced', None]
}

lr_grid = GridSearchCV(
    LogisticRegression(max_iter=1000),
    lr_param_grid,
    cv=5,
    scoring='recall',
    n_jobs=-1,
    verbose=1
)
lr_grid.fit(X_train_res, y_train_res)


In [ ]:
# apply gridsearchcv to random forest
rf_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 15],
    'min_samples_split': [2, 5],
    'class_weight': ['balanced']
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    rf_param_grid,
    cv=5,
    scoring='recall',
    n_jobs=-1,
    verbose=1
)
rf_grid.fit(X_train_res, y_train_res)

In [ ]:
# apply gridsearchcv to XGBoost optimization with medical focus
xgb_param_grid = {
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 5],
    'n_estimators': [100, 200],
    'scale_pos_weight': [10, 15]
}

xgb_grid = GridSearchCV(
    XGBClassifier(random_state=42, eval_metric='logloss'),
    xgb_param_grid,
    cv=5,
    scoring='recall',
    n_jobs=-1
)
xgb_grid.fit(X_train_res, y_train_res)

In [ ]:
# Extract and analyze optimization results
print("Optimization Results:")
print(f"Logistic Regression Best Parameters: {lr_grid.best_params_}")
print(f"Random Forest Best Parameters: {rf_grid.best_params_}")
print(f"XGBoost Best Parameters: {xgb_grid.best_params_}")

print(f"\nLogistic Regression Best Recall: {lr_grid.best_score_:.3f}")
print(f"Random Forest Best Recall: {rf_grid.best_score_:.3f}")
print(f"XGBoost Best Recall: {xgb_grid.best_score_:.3f}")

In [ ]:
# Create final models with best parameters
final_lr = lr_grid.best_estimator_
final_rf = rf_grid.best_estimator_
final_xgb = xgb_grid.best_estimator_

print("Final optimized models ready for evaluation")
print(f"Logistic Regression: {type(final_lr).__name__}")
print(f"Random Forest: {type(final_rf).__name__}")
print(f"XGBoost: {type(final_xgb).__name__}")

# Store for Section 6
optimized_models = {
    'Logistic Regression': final_lr,
    'Random Forest': final_rf,
    'XGBoost': final_xgb
}

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def evaluate_model(y_true, y_pred, y_pred_proba, model_name):
    print(f"\n{model_name} Evaluation metrics:")
    print("Accuracy:", round(accuracy_score(y_true, y_pred), 4))
    print("Precision:", round(precision_score(y_true, y_pred), 4))
    print("Recall:", round(recall_score(y_true, y_pred), 4))
    print("F1 Score:", round(f1_score(y_true, y_pred), 4))
    print("ROC-AUC (Probability):", round(roc_auc_score(y_true, y_pred_proba), 4))

# Get probability predictions for ROC-AUC
y_pred_proba_lr = final_lr.predict_proba(X_test)[:, 1]
y_pred_proba_rf = final_rf.predict_proba(X_test)[:, 1]
y_pred_proba_xgb = final_xgb.predict_proba(X_test)[:, 1]

# Get binary predictions
y_pred_lr = final_lr.predict(X_test)
y_pred_rf = final_rf.predict(X_test)
y_pred_xgb = final_xgb.predict(X_test)

# Evaluate all models with enhanced function
evaluate_model(y_test, y_pred_lr, y_pred_proba_lr, "Logistic Regression")
evaluate_model(y_test, y_pred_rf, y_pred_proba_rf, "Random Forest")
evaluate_model(y_test, y_pred_xgb, y_pred_proba_xgb, "XGBoost Classifier")

In [ ]:
results = {
    "Model": ["Logistic Regression", "Random Forest", "XGBoost"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_xgb),
    ],
    "Precision": [
        precision_score(y_test, y_pred_lr),
        precision_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_xgb)
    ],
    "Recall": [
        recall_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_rf),
        recall_score(y_test, y_pred_xgb)
    ],
    "F1 Score": [
        f1_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_xgb)
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, y_pred_proba_lr),
        roc_auc_score(y_test, y_pred_proba_rf),
        roc_auc_score(y_test, y_pred_proba_xgb)
    ]
}

results_df = pd.DataFrame(results)
results_df.set_index("Model", inplace=True)
print("\nModel Performance Summary:\n")
print(results_df.round(4))

# Bar chart
results_df.plot(kind='bar', figsize=(10,6), colormap='viridis')
plt.title("Model Performance Comparison")
plt.ylabel("Score")
plt.ylim(0.1, 1.0)
plt.xticks(rotation=15)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, roc_curve

models = {
    "Logistic Regression": y_pred_lr,
    "Random Forest": y_pred_rf,
    "XGBoost": y_pred_xgb
}

plt.figure(figsize=(14, 4))
for i, (name, y_pred) in enumerate(models.items(), 1):
    plt.subplot(1, 3, i)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title(f"{name}\nConfusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
plt.tight_layout(pad=2.0)
plt.show()

In [ ]:
plt.figure(figsize=(7, 6))
for name in models.keys():
    if name == "Logistic Regression":
        y_proba = y_pred_proba_lr
    elif name == "Random Forest":
        y_proba = y_pred_proba_rf
    else:
        y_proba = y_pred_proba_xgb

    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})")

plt.plot([0, 1], [0, 1], 'k--', label="Random Guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison (Probability-Based)")
plt.legend()
plt.show()

In [ ]:
from xgboost.plotting import plot_importance

plt.figure(figsize=(10,8))
plot_importance(final_xgb, max_num_features=10)
plt.title("Top 10 Most Important Factors (XGBoost)")
plt.show()